# A 2D Prinz extension: trajectories, MSMs, eigenmodes and relaxation
This is a **separable teaching extension**, not a published canonical 2D Prinz potential:
$$V_{2D}(x,y)=V(x)+V(y),\qquad V(x)=2x^8+0.8e^{-80x^2}+0.2e^{-80(x-0.5)^2}+0.5e^{-40(x+0.5)^2}.$$
There are 16 wells on $[-1,1]^2$. We use a reversible Metropolis chain on a fixed 40×40 grid, $\beta=5$, with reflecting-by-rejection boundaries. Each MC step attempts at most one nearest-neighbor move; time is **MC steps**, not ns. This grid and its time scale differ from the 160-point 1D lesson.

We will plot the surface and populations, watch one particle, estimate an MSM from independent trajectories, visualize its leading modes, and propagate an ensemble initially confined to one MSM state. The single-particle occupancy histogram is a time average, not the instantaneous ensemble distribution.

The model is deliberately separable: exact reference calculations are tractable. Symmetry can produce degenerate mode pairs; arbitrary rotated combinations in a degenerate subspace are equally valid.

[Open in Google Colab](https://colab.research.google.com/github/paulrobustelli/CSMP_150_2026/blob/main/notebooks/Prinz_2D_MSM.ipynb)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.sparse.csgraph import connected_components
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import display, Video
from pathlib import Path
np.set_printoptions(precision=5,suppress=True)
def potential(x):
    return 2*x**8+.8*np.exp(-80*x*x)+.2*np.exp(-80*(x-.5)**2)+.5*np.exp(-40*(x+.5)**2)
N=40;beta=5.;q=.25
x=np.linspace(-1,1,N);X,Y=np.meshgrid(x,x);energy=potential(X)+potential(Y)
w1=np.exp(-beta*potential(x));w1/=w1.sum()
W=np.outer(w1,w1);w=W.ravel()
P1=np.zeros((N,N))
for i in range(N):
    for j in [i-1,i+1]:
        if 0<=j<N:P1[i,j]=q*min(1.,np.exp(-beta*(potential(x[j])-potential(x[i]))))
    P1[i,i]=1-P1[i].sum()
# P2D = (P1 tensor I + I tensor P1)/2; avoid a dense 1600-square matrix.
# Flattening is y-major: state = y_index*N + x_index.
np.testing.assert_allclose(w1[:,None]*P1,w1[None,:]*P1.T,atol=1e-14)
output_dir=Path(".")


## Potential energy surface and equilibrium population
$\mu(x,y)\propto e^{-\beta V_{2D}(x,y)}$. The heatmap below shows **grid-cell probability**, summing to one. Energy and probability use separate color scales. The lines at −0.5, 0 and 0.5 identify the 16 wells.

In [ ]:
fig=plt.figure(figsize=(13,4.5),layout="constrained")
a=fig.add_subplot(131,projection="3d")
a.plot_surface(X,Y,energy,cmap="viridis",linewidth=0);a.set(xlabel="x",ylabel="y",zlabel="V(x,y)",title="2D potential surface")
b=fig.add_subplot(132);im=b.pcolormesh(X,Y,energy,cmap="viridis",shading="nearest");fig.colorbar(im,ax=b,label="Potential energy")
b.set(title="Potential energy landscape",xlabel="x",ylabel="y",aspect="equal")
d=fig.add_subplot(133);im=d.pcolormesh(X,Y,W,cmap="magma",shading="nearest");fig.colorbar(im,ax=d,label="Probability per grid cell")
d.set(title="Equilibrium population",xlabel="x",ylabel="y",aspect="equal")
for ax in [b,d]:
    for edge in [-.5,0,.5]:ax.axvline(edge,color="white",ls="--",lw=.7);ax.axhline(edge,color="white",ls="--",lw=.7)
fig.savefig(output_dir/"Prinz_2D_landscape.png",dpi=180);plt.show()


## Generate trajectories from the fixed reference chain
Choose an axis with probability 1/2, then take one step of the 1D chain on that axis. Rejections and self-transitions are recorded. Independent seeds define separate training and validation paths. We start training at equilibrium so the symmetrized-count estimator below has the intended equilibrium sampling interpretation.

In [ ]:
def simulate_2d(steps,seed,start=None):
    rng=np.random.default_rng(seed)
    state=int(rng.choice(N*N,p=w)) if start is None else int(start)
    path=np.empty(steps+1,dtype=np.int16);path[0]=state
    axes=rng.integers(0,2,size=steps);draws=rng.random(steps)
    left=np.r_[0,np.diag(P1,-1)];right=np.r_[np.diag(P1,1),0]
    iy,ix=divmod(state,N)
    for t in range(steps):
        pos=ix if axes[t]==0 else iy
        rnd=draws[t]
        move=-1 if rnd<left[pos] else (1 if rnd<left[pos]+right[pos] else 0)
        if axes[t]==0:ix+=move
        else:iy+=move
        path[t+1]=iy*N+ix
    return path
STEPS=500_000
training=[simulate_2d(STEPS,s) for s in range(4100,4108)]
validation=[simulate_2d(STEPS,s) for s in range(5100,5104)]
for path in training+validation:
    iy,ix=np.divmod(path.astype(int),N)
    assert np.all(np.abs(np.diff(ix))+np.abs(np.diff(iy))<=1)
observed=np.bincount(np.concatenate(training),minlength=N*N).reshape(N,N).astype(float)
observed/=observed.sum()
fig,axes=plt.subplots(1,3,figsize=(12,4),layout="constrained")
for ax,z,title in zip(axes,[W,observed,observed-W],["Equilibrium","Observed training population","Observed − equilibrium"]):
    lim=np.max(abs(z))
    im=ax.imshow(z,origin="lower",extent=(-1,1,-1,1),cmap="coolwarm" if "−" in title else "magma",vmin=-lim if "−" in title else 0,vmax=lim)
    ax.set(title=title,xlabel="x",ylabel="y");fig.colorbar(im,ax=ax)
fig.savefig(output_dir/"Prinz_2D_populations.png",dpi=180);plt.show()
print("Training MC steps:",len(training)*STEPS,"Validation MC steps:",len(validation)*STEPS)


## Estimate a spatial MSM
Use an 8×8 partition (64 states) that refines the 16 wells. Row $i$ of $T$ contains destination probabilities given starting state $i$. At lag $\tau=100$ count all within-trajectory pairs. Fit $F=(C+C^T)/2$, $T_{ij}=F_{ij}/\sum_jF_{ij}$ and $\pi_i=\sum_jF_{ij}/\sum_{ij}F_{ij}$. This transparent reversible estimator is not a reversible maximum-likelihood estimator. Do not add pseudocounts to hide disconnected sampling.

In [ ]:
B=8;M=B*B;lag=100
assignment=((np.arange(N)[:,None]*B//N)*B+np.arange(N)[None,:]*B//N).ravel()
def counts_at(paths,tau):
    C=np.zeros((M,M),dtype=np.int64)
    for path in paths:
        labels=assignment[path]
        C+=np.bincount(labels[:-tau]*M+labels[tau:],minlength=M*M).reshape(M,M)
    return C
def fit(C):
    F=(C+C.T)/2;mass=F.sum(1)
    if np.any(mass==0) or connected_components(F>0,directed=False,return_labels=False)!=1:
        raise ValueError("Insufficient sampling: extend trajectories; do not silently delete states")
    return F/mass[:,None],mass/mass.sum()
C=counts_at(training,lag);T,pi=fit(C)
np.testing.assert_allclose(T.sum(1),1,atol=1e-12)
np.testing.assert_allclose(pi[:,None]*T,pi[None,:]*T.T,atol=1e-12)
print("Minimum outgoing lagged count:",C.sum(1).min())
print("Count graph connected; detailed balance and probability conservation pass.")
fig,ax=plt.subplots(figsize=(5,4.5),layout="constrained")
ax.contourf(X,Y,energy,25,cmap="viridis")
for edge in np.linspace(-1,1,B+1):ax.axvline(edge,color="white",lw=.5);ax.axhline(edge,color="white",lw=.5)
ax.set(title="64 MSM states over 16 wells",xlabel="x",ylabel="y",aspect="equal");plt.show()


## Dominant eigenmodes and implied timescales
Diagonalize the symmetric matrix $S=D_\pi^{1/2}TD_\pi^{-1/2}$. If $Sv_i=\lambda_i v_i$, then $\psi_i=D_\pi^{-1/2}v_i$, normalized by $\psi_i^TD_\pi\psi_j=\delta_{ij}$.

Plot each mode on the spatial partition. Red/blue indicate signs, not probabilities or fixed flow directions. The initial amplitude determines which sign is excess population. The stationary mode is constant. For positive nonstationary eigenvalues, $t_i=-\tau/\log\lambda_i$.

Because x and y have identical potentials, pairs of exact modes can have the same eigenvalue. Sampling can rotate and split these pairs; compare their subspaces rather than insisting on one orientation.

In [ ]:
def modes(T,pi):
    root=np.sqrt(pi);S=root[:,None]*T/root[None,:]
    np.testing.assert_allclose(S,S.T,atol=1e-12)
    vals,U=eigh(S);vals=vals[::-1];psi=U[:,::-1]/root[:,None]
    if psi[:,0].sum()<0:psi[:,0]*=-1
    return vals,psi
vals,psi=modes(T,pi)
np.testing.assert_allclose(T@psi,psi*vals,atol=1e-11)
np.testing.assert_allclose(psi.T@(pi[:,None]*psi),np.eye(M),atol=1e-11)
fig,axes=plt.subplots(2,3,figsize=(11,7),layout="constrained")
for i,ax in enumerate(axes.ravel()):
    field=psi[:,i].reshape(B,B);lim=np.max(abs(field))
    im=ax.imshow(field,origin="lower",extent=(-1,1,-1,1),cmap="coolwarm",vmin=-lim,vmax=lim)
    ax.set(title=f"Mode {i+1}; eigenvalue {vals[i]:.4f}",xlabel="x",ylabel="y");fig.colorbar(im,ax=ax)
fig.savefig(output_dir/"Prinz_2D_eigenmodes.png",dpi=180);plt.show()
lags=[25,50,100,200,500,1000];times=[]
for tau in lags:
    fitted,stationary=fit(counts_at(training,tau));v,_=modes(fitted,stationary)
    times.append([-tau/np.log(z) if 0<z<1 else np.nan for z in v[1:6]])
fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
for i in range(5):ax.plot(lags,np.array(times)[:,i],"o-",label=f"Mode {i+2}")
ax.set(xlabel="Lag (MC steps)",ylabel="Implied timescale (MC steps)",title="New count fit at every lag")
ax.legend();plt.show()


## Put all probability in one state, then propagate
Choose one interior MSM cell, $s_0$, and set $p_{0,s}=\delta_{s,s_0}$. This is a probability distribution over cells, not a single particle trajectory.
$$u_0=p_0/\pi,\qquad a_i=\sum_s p_{0,s}\psi_i(s)=\psi_i(s_0),$$
$$u_k(s)=\sum_{i=1}^{M} a_i\lambda_i^k\psi_i(s),\qquad p_k(s)=\pi_su_k(s)=\big[p_0T^k\big]_s.$$
All 64 modes are used to reconstruct the sharply localized starting distribution; a handful of slow modes alone need not reconstruct it or preserve positivity at early times. At late times $p_k\to\pi$. The movie shows the first five nonstationary amplitudes, but propagates with the complete matrix.

In [ ]:
initial_state=2*B+2
p0=np.zeros(M);p0[initial_state]=1
amplitudes=p0@psi
steps=300
population=np.empty((steps+1,M));population[0]=p0
for k in range(steps):population[k+1]=population[k]@T
spectral=np.array([pi*(psi@(amplitudes*vals**k)) for k in range(steps+1)])
np.testing.assert_allclose(spectral,population,atol=1e-10)
np.testing.assert_allclose(population.sum(1),1,atol=1e-12)
assert population.min()>-1e-12
TV=.5*np.abs(population-pi).sum(1)
print("Starting state:",initial_state,"(x bin, y bin)",(initial_state%B,initial_state//B))
print("First six initial amplitudes:",amplitudes[:6])
print("Final total variation from fitted equilibrium:",TV[-1])
fig,axes=plt.subplots(1,5,figsize=(15,3.5),layout="constrained")
for ax,k in zip(axes,[0,1,10,50,300]):
    im=ax.imshow(population[k].reshape(B,B),origin="lower",extent=(-1,1,-1,1),cmap="magma",vmin=0,vmax=1)
    ax.set(title=f"k={k}; {k*lag:,} MC steps",xlabel="x",ylabel="y")
fig.colorbar(im,ax=axes,label="Probability per MSM state (fixed scale)")
fig.savefig(output_dir/"Prinz_2D_relaxation.png",dpi=180);plt.show()


## Check the fitted model against held-out trajectories
The propagation above is exactly correct **for the fitted MSM**. It is not automatically exact for the fine-grid dynamics. Compare direct held-out return probabilities with powers of the lag-100 MSM for four representative cells. This is a diagnostic, not a claim that a small subset establishes all-transition accuracy. Extend it to the full matrix and add whole-trajectory uncertainty as an exercise.

In [ ]:
K=np.arange(11);chosen=[2*B+2,2*B+5,5*B+2,5*B+5]
fig,axes=plt.subplots(1,4,figsize=(13,3.5),layout="constrained")
for s,ax in zip(chosen,axes):
    pred=[];direct=[]
    for k in K:
        pred.append(np.linalg.matrix_power(T,int(k))[s,s])
        if k==0:direct.append(1.)
        else:
            counts=counts_at(validation,int(k*lag));direct.append(counts[s,s]/counts[s].sum())
    ax.plot(K,pred,label="MSM");ax.plot(K,direct,"k.",label="Held-out")
    ax.set(title=f"State {s}",xlabel="Lag steps k",ylim=(0,1))
axes[0].set_ylabel("Return probability");axes[0].legend();plt.show()


## Export two distinct movies
The MC movie shows a local path over the energy contours, a moving point on the 3D potential surface at $z=V(x,y)$, and the running occupancy histogram. It displays every fifth MC step and labels that stride; the histogram counts all steps. The ensemble movie shows state populations, excess/depleted probability $p_k-\pi$, five mode amplitudes, and distance to equilibrium. Population color limits stay fixed; a power-law color normalization makes small populations visible without changing the data. The signed excess map uses a fixed symmetric-log color scale to reveal small deviations.

In [ ]:
# imageio-ffmpeg supplies a portable encoder on local Python and Colab.
import importlib.util,subprocess,sys
if importlib.util.find_spec("imageio_ffmpeg") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","imageio-ffmpeg"])
import imageio_ffmpeg
plt.rcParams["animation.ffmpeg_path"]=imageio_ffmpeg.get_ffmpeg_exe()
from matplotlib.colors import PowerNorm, SymLogNorm
EXPORT_MOVIES=True
if EXPORT_MOVIES:
    movie_path=simulate_2d(1500,6100,start=(N//3)*N+N//3)
    fig=plt.figure(figsize=(13,4.3),layout="constrained")
    ax=fig.add_subplot(131,projection="3d",computed_zorder=False);ax.plot_surface(X,Y,energy,cmap="viridis",alpha=.65,linewidth=0,zorder=1)
    ball,=ax.plot([],[],[],"o",color="red",ms=8,zorder=10);ax.set(xlabel="x",ylabel="y",zlabel="V",title="Particle on potential surface")
    xy=fig.add_subplot(132);xy.contourf(X,Y,energy,25,cmap="viridis")
    trail,=xy.plot([],[],color="white",lw=1);dot,=xy.plot([],[],"ro",ms=6)
    xy.set(xlabel="x",ylabel="y",aspect="equal")
    histax=fig.add_subplot(133)
    hist=histax.imshow(np.zeros((N,N)),origin="lower",extent=(-1,1,-1,1),cmap="magma",norm=PowerNorm(.5,vmin=0,vmax=1))
    histax.contour(X,Y,W,levels=5,colors="cyan",linewidths=.5)
    histax.set(xlabel="x",ylabel="y",title="Running occupancy")
    fig.colorbar(hist,ax=histax,label="Probability per grid cell")
    def update_mc(frame):
        t=frame*5;iy,ix=divmod(int(movie_path[t]),N)
        ball.set_data_3d([x[ix]],[x[iy]],[energy[iy,ix]])
        recent=movie_path[max(0,t-100):t+1];yy,xx=np.divmod(recent,N)
        trail.set_data(x[xx],x[yy]);dot.set_data([x[ix]],[x[iy]])
        hist.set_data(np.bincount(movie_path[:t+1],minlength=N*N).reshape(N,N)/(t+1))
        xy.set_title(f"MC step {t}; 5 steps/frame")
    movie_mc=FuncAnimation(fig,update_mc,frames=301,interval=70)
    movie_mc.save(str(output_dir/"Prinz_2D_MC.mp4"),writer=FFMpegWriter(fps=15),dpi=110);plt.close(fig)
    fig,axes=plt.subplots(2,2,figsize=(10,8),layout="constrained")
    popim=axes[0,0].imshow(population[0].reshape(B,B),origin="lower",extent=(-1,1,-1,1),cmap="magma",norm=PowerNorm(.3,vmin=0,vmax=1))
    fig.colorbar(popim,ax=axes[0,0],label="Probability per state")
    excess=axes[0,1].imshow((population[0]-pi).reshape(B,B),origin="lower",extent=(-1,1,-1,1),cmap="coolwarm",norm=SymLogNorm(linthresh=.001,vmin=-1,vmax=1))
    fig.colorbar(excess,ax=axes[0,1],label="p − equilibrium (symmetric log colors)")
    bars=axes[1,0].bar(np.arange(2,7),amplitudes[1:6])
    lim=max(abs(amplitudes[1:6]))*1.15
    axes[1,0].set(ylim=(-lim,lim),xlabel="Mode",ylabel="Signed amplitude",title="First five nonstationary modes")
    axes[1,0].axhline(0,color="gray",lw=.7)
    axes[1,1].plot(np.arange(steps+1)*lag,TV,color="gray");marker,=axes[1,1].plot([0],[TV[0]],"ro")
    axes[1,1].set(xlabel="MC steps",ylabel="Total variation distance",ylim=(0,1),title="Approach to fitted equilibrium")
    for ax in axes[0]:ax.set(xlabel="x",ylabel="y")
    def update_ensemble(k):
        popim.set_data(population[k].reshape(B,B));excess.set_data((population[k]-pi).reshape(B,B))
        axes[0,0].set_title(f"Population: k={k}; {k*lag:,} MC steps")
        axes[0,1].set_title("Excess (red) / depleted (blue)")
        for i,bar in enumerate(bars,1):bar.set_height(amplitudes[i]*vals[i]**k)
        marker.set_data([k*lag],[TV[k]])
    movie_ensemble=FuncAnimation(fig,update_ensemble,frames=range(steps+1),interval=70)
    movie_ensemble.save(str(output_dir/"Prinz_2D_ensemble_relaxation.mp4"),writer=FFMpegWriter(fps=15),dpi=110);plt.close(fig)
    display(Video(str(output_dir/"Prinz_2D_MC.mp4"),embed=True))
    display(Video(str(output_dir/"Prinz_2D_ensemble_relaxation.mp4"),embed=True))


## Student investigations
1. Extend the MC trajectory budget and compare 4×4, 8×8 and finer spatial MSMs, keeping the 40×40 underlying chain fixed. Modify the partition assignment and recompute M, counts, fits and all downstream quantities.
2. Visualize the leading modes, identify approximate x/y exchange pairs, and compare implied timescales from independently fitted lags.
3. Start the ensemble in different individual MSM states. Calculate $a_i=\psi_i(s_0)$ and compare relaxation times and mode contributions.
4. Compare the complete reconstruction with truncation to a few slow modes. Check normalization and negative probabilities, especially at early times.
5. Validate all transition probabilities with held-out data and whole-trajectory uncertainty. Explain discrepancies due to coarse states versus finite sampling.
6. Optional: add a clearly stated coupling term to the potential. Rebuild the sampler and equilibrium weights consistently; the separable reference formulas no longer apply.

**Scope:** a reversible discrete MC teaching model, not molecular dynamics or physical-time kinetics. The 1D shape follows the lecture's Prinz-style convention. The 2D sum construction is an instructional extension.